# Hiltunen Growth and Biochemistry Data Exploration

This Phase 2 notebook inspects the supplied Excel workbook without changing or saving raw data. It reports workbook sheets, dataset structure, missing values, duplicate rows, and candidate variables. No preprocessing or model training is performed.

In [3]:
from pathlib import Path
import re

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

print(
    "Library versions:",
    {
        "pandas": pd.__version__,
        "NumPy": np.__version__,
        "matplotlib": matplotlib.__version__,
        "seaborn": sns.__version__,
    },
)
print("Matplotlib backend:", plt.get_backend())

WORKBOOK_RELATIVE_PATH = Path("data/raw/hiltunen/hiltunen_etal_chlorella_growth_biochem_data.xlsx")
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (location for location in search_locations if (location / WORKBOOK_RELATIVE_PATH).is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find {WORKBOOK_RELATIVE_PATH} from the current directory or its parents."
    )

workbook_path = PROJECT_ROOT / WORKBOOK_RELATIVE_PATH
daily_cell_pattern = re.compile(r"^celldens_d(\d+)$", flags=re.IGNORECASE)
daily_ph_pattern = re.compile(r"^ph_d(\d+)$", flags=re.IGNORECASE)

with pd.ExcelFile(workbook_path) as workbook:
    sheet_names = workbook.sheet_names
    print("Workbook sheets:", sheet_names)
    sheet_frames = {sheet_name: workbook.parse(sheet_name) for sheet_name in sheet_names}

def daily_column_count(columns, pattern):
    return sum(pattern.fullmatch(str(column)) is not None for column in columns)

sheet_scores = {
    sheet_name: daily_column_count(frame.columns, daily_cell_pattern)
    + daily_column_count(frame.columns, daily_ph_pattern)
    for sheet_name, frame in sheet_frames.items()
}
main_sheet_name = max(sheet_scores, key=sheet_scores.get)
if sheet_scores[main_sheet_name] == 0:
    raise ValueError(
        "No sheet contains daily cell-density or pH columns matching the expected day-index pattern."
    )

data = sheet_frames[main_sheet_name]
print(f"\nMain data sheet: {main_sheet_name!r}")
print(f"Dimensions: {data.shape[0]} rows x {data.shape[1]} columns")
print("\nFirst five rows:")
display(data.head())
print("\nColumn names:")
print(data.columns.tolist())
print("\nData types:")
print(data.dtypes.to_string())
print("\nMissing-value counts by column (including zero counts):")
print(data.isna().sum().to_string())
print(f"\nDuplicate rows: {data.duplicated().sum()}")

variable_patterns = {
    "experiment": r"experiment|^exp$|funnel",
    "CO2": r"co.?2|carbon dioxide",
    "medium": r"medium",
    "light": r"light",
    "cell density": r"cell.?dens",
    "pH": r"^ph(?:_|$)",
    "nutrients": r"po4|phosphate|no3|nitrate|tn_|nitrogen|nutrient",
    "biomass": r"biomass|dry.?weight|(?:^|_)dw(?:_|$)|afdw|chla|productiv",
    "DIC": r"dic|dissolved inorganic carbon",
}
print("\nCandidate columns by requested variable group:")
for group_name, pattern in variable_patterns.items():
    matches = [
        str(column)
        for column in data.columns
        if re.search(pattern, str(column), flags=re.IGNORECASE)
    ]
    print(f"{group_name}: {matches}")

daily_cell_columns = sorted(
    (str(column) for column in data.columns if daily_cell_pattern.fullmatch(str(column))),
    key=lambda column: int(daily_cell_pattern.fullmatch(column).group(1)),
)
daily_ph_columns = sorted(
    (str(column) for column in data.columns if daily_ph_pattern.fullmatch(str(column))),
    key=lambda column: int(daily_ph_pattern.fullmatch(column).group(1)),
)
print("\nDaily cell-density columns in day order:", daily_cell_columns)
print("Daily pH columns in day order:", daily_ph_columns)

Library versions: {'pandas': '3.0.6', 'NumPy': '2.5.3', 'matplotlib': '3.11.2', 'seaborn': '0.13.2'}
Matplotlib backend: module://matplotlib_inline.backend_inline
Workbook sheets: ['info', 'data']

Main data sheet: 'data'
Dimensions: 61 rows x 133 columns

First five rows:


,Exp,Funnel#,Medium,CO2,Light,celldens_d0,celldens_d1,celldens_d2,celldens_d3,celldens_d4,...,zeaxanthin_exp,a-carotene_exp,b-carotene_exp,total_carotenoids_µg/mgDW_stat,violaxanthin_stat,neoxanthin_stat,lutein_stat,zeaxanthin_stat,a-carotene_stat,b-carotene_stat
0,1,9,RAS,RASair,low,3.44,3.58,3.82,5.01,6.73,...,1.179,7.074,11.860,9.937,15.020,19.767,50.060,1.884,5.662,7.606
1,1,10,RAS,RoomAir,low,3.44,3.41,3.55,4.59,5.88,...,0.779,6.431,11.738,10.614,14.491,15.049,52.147,2.330,6.580,9.403
2,1,11,RAS,RASair,low,3.44,3.58,4.11,5.85,10.30,...,0.597,6.940,10.150,6.874,15.471,22.996,47.314,2.010,5.589,6.619
3,1,12,MWC,RASair,low,3.43,3.54,3.62,4.62,6.20,...,0.362,7.231,11.200,6.471,13.747,17.297,55.747,4.737,1.353,7.119
4,1,13,MWC,RoomAir,low,3.43,3.75,3.97,4.49,6.01,...,0.354,7.392,11.855,6.096,13.950,17.017,55.696,3.472,1.773,8.091



Column names:
['Exp', 'Funnel#', 'Medium', 'CO2', 'Light', 'celldens_d0', 'celldens_d1', 'celldens_d2', 'celldens_d3', 'celldens_d4', 'celldens_d5', 'celldens_d6', 'celldens_d7', 'celldens_d8', 'celldens_d9', 'celldens_d10', 'celldens_d11', 'celldens_d12', 'celldens_d13', 'celldens_d14', 'celldens_d15', 'celldens_d16', 'pH_d0', 'pH_d1', 'pH_d2', 'pH_d3', 'pH_d4', 'pH_d5', 'pH_d6', 'pH_d7', 'pH_d8', 'pH_d9', 'pH_d10', 'pH_d11', 'pH_d12', 'pH_d13', 'pH_d14', 'pH_d15', 'pH_d16', 'DIC_exp', 'DIC_stat', 'chla_mg/L', 'DW_mg/L', 'AFDW_mg/L', 'PO4-P _mg/L_start', 'PO4-P_mg/L_end', 'PO4-P_removal', 'NO3-N_mg/L_start', 'NO3-N_mg/L_end', 'NO3-N_removal', 'TN_mg/L_start', 'TN_mg/L_end', 'TN_removal', 'DOC_mg/L_start', 'DOC_mg/L_end', 'd13C', 'd15N', 'C%', 'N%', 'mass_CN', 'molar_CN', 'Lipid%_exp', 'Total_FA_µg/mgDW_exp', '12:0_exp', '14:0_exp', '16:0_exp', '16:1w9_exp', '16:1w7_exp', '16:1w5_exp', '16:2w6_exp', '17:1_exp', '16:3w3_exp', '18:0_exp', '18:1w9_exp', '18:1w7_exp', '18:2w6_exp', '18:3w